# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all data entities by their `@id` fields in alignment with the Croissant schema.

### Dataset Source
The dataset is described using a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print("Dataset metadata loaded.")
print(f"Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and columns with their `@id` attributes, as defined by the Croissant schema.

In [ ]:
# Explore the dataset to identify record sets and their fields/columns
record_sets = list(dataset.record_sets)
print("Available record sets (by @id):")
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for field in rs.fields:
            print(f"      - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    if hasattr(rs, 'columns') and rs.columns:
        print("    Columns:")
        for col in rs.columns:
            print(f"      - @id: {col.id}, name: {col.name}, dataType: {col.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field/column `@id`s from the overview to specify which portions of the data to extract.

In [ ]:
# Select available record set @ids for extraction
rs_ids = [rs.id for rs in record_sets]
dataframes = {}

print("Extracting records for each record set...")
for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} records from {rs_id}")

# Pick the first record set for demonstration (replace with the appropriate @id if needed)
main_rs_id = rs_ids[0] if rs_ids else None
if main_rs_id:
    print(f"Columns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All operations reference fields using their `@id`.

In [ ]:
import numpy as np
# Identify a numeric field from the list of fields previously printed, for demonstration e.g., '@id': 'accession_coverage_percent'
# You may have to update 'coverage_percent' to the actual numeric field's @id present in your dataset.

numeric_field_id = None
group_field_id = None

# Attempt to auto-detect a likely numeric field by looking for 'percent', 'count', 'mw', 'abundance', etc.
if main_rs_id:
    possible_numeric_patterns = ['coverage', 'count', 'mw', 'abundance', 'peptide', 'score']
    for col in dataframes[main_rs_id].columns:
        for pat in possible_numeric_patterns:
            if pat in col.lower():
                numeric_field_id = col
                break
        if numeric_field_id:
            break
    if numeric_field_id is None and len(dataframes[main_rs_id].columns):
        # fallback to first column
        numeric_field_id = dataframes[main_rs_id].columns[0]
    # Try to find a grouping field (e.g., a categorical field)
    cat_patterns = ['sample', 'type', 'experiment', 'group', 'class', 'species']
    for col in dataframes[main_rs_id].columns:
        for pat in cat_patterns:
            if pat in col.lower():
                group_field_id = col
                break
        if group_field_id:
            break

    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using grouping field: {group_field_id if group_field_id else 'None (not found)'}")

    # Filter rows with value above threshold
    # For demonstration, we assume it's a float; coerce errors to NaN, then dropna
    df = dataframes[main_rs_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].dropna(subset=[numeric_field_id])
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={filtered_df.shape[0]}):")
    display(filtered_df.head())

    # Normalize column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the selected (categorical) field if possible
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No records set available for EDA.")

## 5. Visualization
Visualize distributions and relationships using selected fields. All references use `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in dataframes[main_rs_id].columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_rs_id], showfliers=False)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset using the `mlcroissant` library:
- We loaded the dataset's Croissant schema and reviewed metadata.
- We listed all record sets, fields, and columns, referencing them by `@id` as defined by the schema.
- We extracted records into DataFrames, performed basic filtering and normalization, and visualized data distributions.

This approach enables reproducible, standards-based analysis leveraging the Croissant metadata for robust data interoperability and transparency.

Feel free to extend this notebook for advanced analysis, machine learning modeling, or further integration with the Croissant ecosystem.